# 03 — LFM2.5-8B-A1B (MoE): QLoRA-Finetuning

Dieselbe Kette wie Notebook 01, aber für das **Mixture-of-Experts**-Modell
`LiquidAI/LFM2.5-8B-A1B` — 8.3B Gesamtparameter, 1.5B pro Token aktiv.

**Der Denkfehler, den man hier macht:** der Speicherbedarf richtet sich nach den
*Gesamt*parametern, nicht nach den aktiven. In 4-bit sind das rund 5.5 GB — auf
einer 15-GB-T4 bleibt wenig Luft, auf einer A100/L4 (40 GB) ist es entspannt.

Die Zelle unten prüft die GPU und passt die Defaults an.

In [ ]:
# Colab brings torch; we only need the training stack. --upgrade on purpose:
# LFM2.5 checkpoints need a recent transformers.
!pip install -q -U transformers trl peft accelerate bitsandbytes datasets pyyaml

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/muscal-lfm/moe')
(DRIVE_DIR / 'data').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
print("drive project dir:", DRIVE_DIR)
print("contents:", sorted(p.name for p in DRIVE_DIR.iterdir()))

In [ ]:
# Get the training code. If you already have the repo in Drive, skip this cell.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/INDIEaner84/MUSCAL-ColabAPI-ProviderLLM.git"
REF = os.environ.get("MUSCAL_REF", "main")   # set to your branch before pushing

if not Path('/content/MUSCAL-ColabAPI-ProviderLLM').exists():
    !git clone --depth 1 --branch $REF $REPO_URL /content/MUSCAL-ColabAPI-ProviderLLM

%cd /content/MUSCAL-ColabAPI-ProviderLLM
sys.path.insert(0, '/content/MUSCAL-ColabAPI-ProviderLLM')
print("cwd:", Path.cwd())

In [ ]:
import torch
from muscal_lfm.model import gpu_info

info = gpu_info()
print(info)

VRAM = info.get("vram_gb", 0)
if VRAM and VRAM < 20:
    print("\n[warn] unter 20 GB VRAM -- bleibe bei batch size 1 und 512 Tokens.")
    PRESET = dict(per_device_train_batch_size=1, gradient_accumulation_steps=16, max_length=512)
else:
    print("\n[info] genug VRAM fuer groessere Batches.")
    PRESET = dict(per_device_train_batch_size=4, gradient_accumulation_steps=4, max_length=2048)
print(PRESET)

In [ ]:
from muscal_lfm.config import TrainConfig
from muscal_lfm.model import gpu_info
import yaml

CONFIG = "configs/moe_qlora.yaml"
cfg = TrainConfig.from_yaml(CONFIG)

# --- your settings -------------------------------------------------------
cfg.data.train_file = str(DRIVE_DIR / "data" / "train.jsonl")
cfg.training.output_dir = str(DRIVE_DIR / "outputs" / "moe.yaml")
cfg.data.max_length = PRESET["max_length"]
cfg.training.per_device_train_batch_size = PRESET["per_device_train_batch_size"]
cfg.training.gradient_accumulation_steps = PRESET["gradient_accumulation_steps"]
cfg.training.learning_rate = 1e-4
# ------------------------------------------------------------------------

print("gpu:", gpu_info())
print()
print(cfg.to_yaml())

## Hinweis zu LoRA an MoE-Schichten

`gate_proj` / `up_proj` / `down_proj` matchen per Suffix **jeden Experten** — die
Adapter-Parameterzahl ist deshalb deutlich höher als beim dichten Modell gleicher
Größe. Bei OOM in dieser Reihenfolge runtergehen:

1. `cfg.lora.r` 16 → 8
2. `cfg.data.max_length` halbieren
3. `per_device_train_batch_size` 1 und `gradient_accumulation_steps` erhöhen

In [ ]:
from muscal_lfm import data as data_utils
from datasets import Dataset

ds = data_utils.load_file(cfg.data.train_file)
print("rows:", len(ds), "| columns:", ds.column_names)
print()
print("example row:")
print(ds[0])

problems_exist = False
try:
    data_utils.validate_conversational(ds)
    print("\n[ok] dataset shape looks good")
except data_utils.DatasetError as exc:
    problems_exist = True
    print("\n[problem]", exc)

print("\nstats:", data_utils.dataset_report(ds))

In [ ]:
from muscal_lfm.train import train
from pathlib import Path

Path(cfg.training.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.training.output_dir, "config.resolved.yaml").write_text(cfg.to_yaml())

adapter_dir = train(cfg)
print("adapter:", adapter_dir)
print(sorted(p.name for p in Path(adapter_dir).iterdir()))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(cfg.model.id)
base = AutoModelForCausalLM.from_pretrained(cfg.model.id, dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base, str(adapter_dir))

PROBE = "Erklaere in einem Satz, was dieses Modell macht."

messages = [{"role": "user", "content": PROBE}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("--- BASE ---")
with torch.no_grad():
    out = base.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

print("\n--- FINETUNED ---")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Merge

Bei 8.3B lohnt es sich, den Merge **nicht** in Colab zu machen, wenn du nur einen
Adapter weitergeben willst — lade den Adapter nach Drive und merg lokal mit
`scripts/merge_adapter.py`. Für GGUF braucht es eine aktuelle llama.cpp mit
LFM-MoE-Support.

In [ ]:
from muscal_lfm.export import merge_adapter, export_gguf

merged = merge_adapter(
    cfg.model.id,
    adapter_dir,
    str(Path(cfg.training.output_dir) / "merged"),
    track=cfg.model.track,
)
print("merged:", merged)

# Optional: GGUF for llama.cpp / Ollama / LM Studio.
# Needs a current llama.cpp (LFM2.5 is a young architecture) and ~10 min.
EXPORT_GGUF = False
if EXPORT_GGUF:
    gguf = export_gguf(merged, quant="q4_k_m")
    !cp "$gguf" "$DRIVE_DIR/outputs/"